# RotorQuant KV-cache compression

RotorQuant is also a KV-cache method, not a saved weight format. The RotorQuant report
replaces dense TurboQuant-style rotations with small Clifford/3D rotor blocks,
claiming fewer parameters and faster kernels.

Sources:
- Technical report/site: https://scrya.com/rotorquant
- Code: https://github.com/scrya-com/rotorquant
- TurboQuant paper it builds on: https://arxiv.org/abs/2504.19874

This notebook implements a self-contained 3D-rotor block reference path using Rodrigues
rotations. It saves a compressed KV cache artifact and theoretical packed-memory metrics.
Use the linked repo for CUDA/Metal kernels and llama.cpp experiments.

In [ ]:
!pip -q install -U transformers accelerate sentencepiece safetensors

## Cell 2 — Helpers

In [ ]:

import json
import math
import time
from pathlib import Path

import torch
import torch.nn.functional as F
from transformers import AutoTokenizer, AutoModelForCausalLM


def bytes_to_mib(n):
    return n / (1024 ** 2)


def random_3d_rotations(n_blocks, device, dtype, seed=5678):
    """Build n_blocks independent 3D Rodrigues rotation matrices.""",
    g = torch.Generator(device=device)
    g.manual_seed(seed)
    axis  = torch.randn(n_blocks, 3, generator=g, device=device, dtype=torch.float32)
    axis  = axis / axis.norm(dim=-1, keepdim=True).clamp(min=1e-6)
    angle = torch.rand(n_blocks, generator=g, device=device, dtype=torch.float32) * (2 * math.pi)
    kx, ky, kz = axis.unbind(-1)
    c, s, one_c = torch.cos(angle), torch.sin(angle), 1 - torch.cos(angle)
    r = torch.zeros(n_blocks, 3, 3, device=device, dtype=torch.float32)
    r[:, 0, 0] = c + kx*kx*one_c;  r[:, 0, 1] = kx*ky*one_c - kz*s; r[:, 0, 2] = kx*kz*one_c + ky*s
    r[:, 1, 0] = ky*kx*one_c + kz*s; r[:, 1, 1] = c + ky*ky*one_c;  r[:, 1, 2] = ky*kz*one_c - kx*s
    r[:, 2, 0] = kz*kx*one_c - ky*s; r[:, 2, 1] = kz*ky*one_c + kx*s; r[:, 2, 2] = c + kz*kz*one_c
    return r.to(device=device, dtype=dtype)


def apply_block_rotations(x, rotations, inverse=False):
    orig_shape = x.shape
    dim = orig_shape[-1]
    pad = (-dim) % 3
    if pad:
        x = F.pad(x, (0, pad))
    blocks = x.reshape(*x.shape[:-1], -1, 3)
    r = rotations.transpose(-1, -2) if inverse else rotations
    y = torch.einsum("...bi,bij->...bj", blocks, r)
    y = y.reshape(*x.shape[:-2], -1)
    if pad:
        y = y[..., :dim]
    return y.reshape(orig_shape)


def quantize_rotor_vectors(x, bits, rotations):
    rotated   = apply_block_rotations(x, rotations, inverse=False)
    orig_shape = rotated.shape
    dim        = orig_shape[-1]
    flat       = rotated.reshape(-1, dim)
    qmax       = (2 ** (bits - 1)) - 1
    scale      = flat.abs().amax(dim=-1, keepdim=True).clamp(min=1e-6) / qmax
    q          = torch.round(flat / scale).clamp(-qmax, qmax).to(torch.int8)
    deq_rot    = (q.to(rotated.dtype) * scale).reshape(orig_shape)
    deq        = apply_block_rotations(deq_rot, rotations, inverse=True)
    return {"q": q.cpu(), "scale": scale.cpu().to(torch.float16), "dequantized": deq.to(x.dtype)}


def iter_kv_layers(past_key_values):
    """Iterate over (key, value) tensors regardless of whether past_key_values
    is the old list-of-tuples format or a newer transformers DynamicCache object."""
    if past_key_values is None:
        return
    try:
        # DynamicCache: has .key_cache / .value_cache as lists
        key_cache = past_key_values.key_cache
        value_cache = past_key_values.value_cache
        for k, v in zip(key_cache, value_cache):
            yield k, v
    except AttributeError:
        # Legacy format: list of (key, value) tuples
        for layer in past_key_values:
            if isinstance(layer, (tuple, list)):
                yield layer[0], layer[1]


def kv_cache_bytes(past_key_values):
    total = 0
    for key, value in iter_kv_layers(past_key_values):
        total += key.numel() * key.element_size()
        total += value.numel() * value.element_size()
    return total


def cosine_report(original, reconstructed):
    a   = original.reshape(-1, original.shape[-1]).float()
    b   = reconstructed.reshape(-1, reconstructed.shape[-1]).float()
    cos = F.cosine_similarity(a, b, dim=-1)
    mse = F.mse_loss(b, a).item()
    return float(cos.mean().item()), float(cos.min().item()), float(mse)


def save_json(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2), encoding="utf-8")
    print(json.dumps(payload, indent=2))


## Cell 3 — Config, Model Load, Compress & Save

In [ ]:
MODEL_ID   = "TinyLlama/TinyLlama-1.1B-Chat-v1.0"
OUTPUT_DIR = Path("/content/quant_outputs/rotorquant_kv")
PROMPT     = "RotorQuant compresses the key value cache using compact block rotations. " * 32
BITS       = 3

device = "cuda" if torch.cuda.is_available() else "cpu"
dtype  = torch.float16 if torch.cuda.is_available() else torch.float32

print(f"Device: {device}  |  Model: {MODEL_ID}  |  Bits: {BITS}")

t0 = time.perf_counter()
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, use_fast=True)
model = AutoModelForCausalLM.from_pretrained(MODEL_ID, torch_dtype=dtype).to(device)
model.eval()
load_seconds = time.perf_counter() - t0
print(f"Model loaded in {load_seconds:.1f}s")

# ── Prefill to capture real KV cache ────────────────────────────────────────
inputs = tokenizer(PROMPT, return_tensors="pt").to(device)
print(f"Prompt tokens: {inputs['input_ids'].shape[-1]}")
with torch.no_grad():
    t0 = time.perf_counter()
    outputs = model(**inputs, use_cache=True)
    prefill_seconds = time.perf_counter() - t0
past = outputs.past_key_values
print(f"Prefill done in {prefill_seconds:.2f}s")

head_dim = next(iter_kv_layers(past))[0].shape[-1]
n_blocks  = math.ceil(head_dim / 3)
rotations = random_3d_rotations(n_blocks, device, dtype)

artifact = {"method": "rotorquant_reference", "bits": BITS, "layers": []}
reports  = []
packed_bits = 0

t0 = time.perf_counter()
for layer_idx, (key, value) in enumerate(iter_kv_layers(past)):
    qk = quantize_rotor_vectors(key,   BITS, rotations)
    qv = quantize_rotor_vectors(value, BITS, rotations)
    k_mean, k_min, k_mse = cosine_report(key,   qk["dequantized"].to(device))
    v_mean, v_min, v_mse = cosine_report(value, qv["dequantized"].to(device))
    n_values  = key.numel() + value.numel()
    n_vectors = key.reshape(-1, head_dim).shape[0] + value.reshape(-1, head_dim).shape[0]
    packed_bits += n_values * BITS
    packed_bits += n_vectors * 16  # per-vector fp16 scale
    artifact["layers"].append({
        "key_q":      qk["q"],   "key_scale":   qk["scale"],
        "value_q":    qv["q"],   "value_scale": qv["scale"],
    })
    reports.append({
        "layer": layer_idx,
        "key_cosine_mean":   k_mean, "key_cosine_min":   k_min, "key_mse":   k_mse,
        "value_cosine_mean": v_mean, "value_cosine_min": v_min, "value_mse": v_mse,
    })
quant_seconds = time.perf_counter() - t0

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
torch.save(artifact, OUTPUT_DIR / "compressed_kv_cache.pt")

fp_bytes     = kv_cache_bytes(past)
packed_bytes = math.ceil(packed_bits / 8)
metrics = {
    "method":                       "rotorquant_reference_kv_cache",
    "model_id":                     MODEL_ID,
    "bits":                         BITS,
    "prompt_tokens":                int(inputs["input_ids"].shape[-1]),
    "num_layers":                   len(reports),
    "head_dim":                     head_dim,
    "n_blocks_per_layer":           n_blocks,
    "fp_cache_mib":                 bytes_to_mib(fp_bytes),
    "theoretical_packed_cache_mib": bytes_to_mib(packed_bytes),
    "compression_ratio":            fp_bytes / packed_bytes,
    "prefill_seconds":              round(prefill_seconds, 3),
    "quant_seconds":                round(quant_seconds, 3),
    "artifact_path":                str(OUTPUT_DIR / "compressed_kv_cache.pt"),
    "reports":                      reports,
}
save_json(OUTPUT_DIR / "metrics.json", metrics)

## Cell 4 — (Optional) Clone RotorQuant Official Repo

In [ ]:
# !git clone https://github.com/scrya-com/rotorquant /content/rotorquant
# %cd /content/rotorquant
# !python -m turboquant.benchmark_perplexity --bits 3 4